# 1. Setup

Simple relative paths and notebook switches. GeoPandas is used only for reading GeoJSON files, CRS transformation, area calculation, and plotting. The spatial algorithms themselves are implemented manually!

Run the following Cell once, to create the folders for the data!

In [1]:
import os
import sys
import glob
import csv
import math
import random
from collections import defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from numpy import sqrt, radians, arcsin, sin, cos

ROOT_DIR = Path.cwd()

if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

# 2. Define Folder Paths 
PATH_DATA = ROOT_DIR / 'data'
PATH_RAW = PATH_DATA / 'raw'
PATH_PROCESSED = PATH_DATA / 'processed'
PATH_OUTPUTS = ROOT_DIR / 'outputs'

# Create the necessary directories safely
for p in [PATH_RAW, PATH_PROCESSED, PATH_OUTPUTS]:
    p.mkdir(parents=True, exist_ok=True)

# 3. Define Input File Paths
DATA_BASE_DIR = PATH_RAW / "DE"
GEOJSON_PARKS = PATH_RAW / "park_polygons.geojson"
GEOJSON_CITIES = PATH_RAW / "city_polygons.geojson"

# 4. Define Output File Paths
COMPILED_CSV = PATH_OUTPUTS / "all_flickr_points.csv"
CANDIDATE_CSV = PATH_OUTPUTS / "flickr_bbox_candidates.csv"
FILTERED_CSV = PATH_OUTPUTS / "filtered_flickr_inside_polygons.csv"
NNI_RESULTS_CSV = PATH_OUTPUTS / "nni_results_sampled.csv"
NNI_QUERY_RESULTS_CSV = PATH_OUTPUTS / "nni_results_sampled_query.csv"

'''
ROOT_DIR = Path.cwd().parent

if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

# Relative Paths
PATH_RAW = ROOT_DIR / 'data' / 'raw'
PATH_OUTPUTS   = ROOT_DIR / 'outputs'
PATH_PROCESSED_DATA = ROOT_DIR / 'data' / 'processed' 
DATA_BASE_DIR = "./data/raw/DE"
GEOJSON_PARKS = "./data/raw/park_polygons.geojson"
GEOJSON_CITIES = "./data/raw/city_polygons.geojson"

# Output-Files
COMPILED_CSV = "./outputs/all_flickr_points.csv"
CANDIDATE_CSV = "./outputs/flickr_bbox_candidates.csv"
FILTERED_CSV = "./outputs/filtered_flickr_inside_polygons.csv"
NNI_RESULTS_CSV = "./outputs/nni_results_sampled.csv"
NNI_QUERY_RESULTS_CSV = "./outputs/nni_results_sampled_query.csv"

# Empty Data Folders
PATH_RAW.mkdir(parents=True, exist_ok=True)
PATH_OUTPUTS.mkdir(parents=True, exist_ok=True)
PATH_PROCESSED_DATA.mkdir(parents=True, exist_ok=True)
#(PATH_OUTPUTS / 'figures').mkdir(parents=True, exist_ok=True)
#(PATH_OUTPUTS / 'data').mkdir(parents=True, exist_ok=True)
'''

# Sample-Configuration
USE_DATA_SAMPLE = False
SAMPLE_MAX_FILES = 12
SAMPLE_ROWS_PER_FILE = 5000

# Cache-Schalter: auf True setzen, wenn Outputs neu gebaut werden sollen
REBUILD_COMPILED_CACHE = False
REBUILD_CANDIDATE_CACHE = False
REBUILD_PIP_RESULTS = False
REBUILD_NNI_RESULTS = False

# Spatial-index & NNI Settings
GLOBAL_INDEX_RESOLUTION = 0.05
NNI_INDEX_RESOLUTION = 0.01
NNI_SAMPLE_SIZE = 5000
RANDOM_SEED = 42
CSV_CHUNK_SIZE = 500000

# Prevent DtypeWarnings warnings:
MIXED_TYPE_DTYPES = {"ID": "string", "PhotoID": "string", "Views": "string", "MTags": "string"}

os.makedirs("./outputs", exist_ok=True)

print("Setup ready!")

Setup ready!


# 2. Data Preparation

The Flickr data are loaded recursively from `./data/raw/DE`. All `settings.txt` files are excluded. Coordinates are cleaned, photos are deduplicated by `PhotoID`, and the cleaned point table is cached in `./outputs/all_flickr_points.csv`.

Some descriptive fields, especially `PhotoID`, `Views`, and `MTags`, can look numeric in some rows and textual or empty in others. These fields are therefore read as strings in later cached reads. This avoids Pandas `DtypeWarning` messages and does not affect the spatial algorithms, because only `Latitude` and `Longitude` are used as coordinates.

In [2]:
FLICKR_COLUMNS = [
    "ID", "Latitude", "Longitude", "NAME", "URL", "PhotoID", "Owner", "UserID",
    "DateTaken", "UploadDate", "Views", "Tags", "MTags"
]


def find_flickr_files(base_dir):
    all_txt_files = sorted(glob.glob(os.path.join(base_dir, "**", "*.txt"), recursive=True))
    data_files = [f for f in all_txt_files if os.path.basename(f).lower() != "settings.txt"]
    settings_files = [f for f in all_txt_files if os.path.basename(f).lower() == "settings.txt"]
    return data_files, settings_files


def validate_headers(files):
    header_counts = defaultdict(int)
    for file_path in files:
        with open(file_path, "r", encoding="utf-8", errors="replace", newline="") as f:
            reader = csv.reader(f)
            header = tuple(next(reader))
            header_counts[header] += 1

    print(f"Flickr data files found: {len(files)}")
    print(f"settings.txt Files excluded: {len(settings_files)}")
    print(f"Header-Variants found: {len(header_counts)}")

    for header, count in header_counts.items():
        print(f"{count} Files with header: {header}")
        if list(header) != FLICKR_COLUMNS:
            raise ValueError("Unexpected header in Flickr-Files.")


flickr_files, settings_files = find_flickr_files(DATA_BASE_DIR)
validate_headers(flickr_files)

Flickr data files found: 403
settings.txt Files excluded: 90
Header-Variants found: 1
403 Files with header: ('ID', 'Latitude', 'Longitude', 'NAME', 'URL', 'PhotoID', 'Owner', 'UserID', 'DateTaken', 'UploadDate', 'Views', 'Tags', 'MTags')


In [ ]:
#import pyarrow

In [11]:
def compile_raw_flickr_data(files, output_csv):
    if os.path.exists(output_csv) and not REBUILD_COMPILED_CACHE:
        print(f"Load already compiled data from {output_csv}...")
        return output_csv

    files_to_read = files[:SAMPLE_MAX_FILES] if USE_DATA_SAMPLE else files
    seen_photo_ids = set()
    rows_read = 0
    rows_written = 0
    duplicate_rows = 0
    invalid_rows = 0

    with open(output_csv, "w", encoding="utf-8", newline="") as out_file:
        wrote_header = False

        for i, file_path in enumerate(files_to_read, start=1):
            nrows = SAMPLE_ROWS_PER_FILE if USE_DATA_SAMPLE else None
            df = pd.read_csv(
                file_path,
                usecols=FLICKR_COLUMNS,
                nrows=nrows,
                dtype=MIXED_TYPE_DTYPES,
                on_bad_lines="skip",
                low_memory=False,
                #engine="pyarrow"
            )
            rows_read += len(df)

            df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
            df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")
            valid_coords = df["Latitude"].between(-90, 90) & df["Longitude"].between(-180, 180)
            invalid_rows += int((~valid_coords).sum())
            df = df.loc[valid_coords].copy()

            df["PhotoID"] = df["PhotoID"].astype(str)
            before_file_dedup = len(df)
            df = df.drop_duplicates(subset="PhotoID", keep="first").copy()
            duplicate_rows += before_file_dedup - len(df)

            already_seen = df["PhotoID"].isin(seen_photo_ids)
            duplicate_rows += int(already_seen.sum())
            df = df.loc[~already_seen].copy()
            seen_photo_ids.update(df["PhotoID"].tolist())

            df.to_csv(out_file, index=False, header=not wrote_header)
            wrote_header = True
            rows_written += len(df)

            if i % 25 == 0 or i == len(files_to_read):
                print(f"Processed: {i}/{len(files_to_read)} Files | saved Points: {rows_written:,}")

    print("Compiling finished.")
    print(f"Read rows: {rows_read:,}")
    print(f"Saved rows: {rows_written:,}")
    print(f"Duplicates, dropped after PhotoID-check: {duplicate_rows:,}")
    print(f"Invalid coordinates dropped: {invalid_rows:,}")
    return output_csv


compile_raw_flickr_data(flickr_files, COMPILED_CSV)
pd.read_csv(COMPILED_CSV, nrows=5, dtype=MIXED_TYPE_DTYPES, low_memory=False)

KeyboardInterrupt: 

Test

In [ ]:
def compile_raw_flickr_data(files, output_csv):
    import csv

    if os.path.exists(output_csv) and not REBUILD_COMPILED_CACHE:
        print(f"Lade bereits kompilierte Daten von {output_csv}...")
        return output_csv

    files_to_read = files[:SAMPLE_MAX_FILES] if USE_DATA_SAMPLE else files

    seen_photo_ids = set()

    rows_read = 0
    rows_written = 0
    duplicate_rows = 0
    invalid_rows = 0

    # Much faster buffering for large CSV writes
    with open(output_csv, "w", encoding="utf-8", newline="", buffering=1024 * 1024) as out_file:

        writer = None

        for i, file_path in enumerate(files_to_read, start=1):

            nrows = SAMPLE_ROWS_PER_FILE if USE_DATA_SAMPLE else None

            try:
                df = pd.read_csv(
                    file_path,
                    usecols=FLICKR_COLUMNS,
                    nrows=nrows,
                    dtype=MIXED_TYPE_DTYPES,
                    on_bad_lines="skip",
                    low_memory=False,
                    engine="pyarrow",   # usually MUCH faster
                )

            except Exception:
                # fallback if pyarrow fails on malformed files
                df = pd.read_csv(
                    file_path,
                    usecols=FLICKR_COLUMNS,
                    nrows=nrows,
                    dtype=MIXED_TYPE_DTYPES,
                    on_bad_lines="skip",
                    low_memory=False,
                )

            rows_read += len(df)

            # Fast numeric conversion
            df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
            df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")

            # Filter invalid coordinates WITHOUT .copy()
            valid_coords = (
                df["Latitude"].between(-90, 90)
                & df["Longitude"].between(-180, 180)
            )

            invalid_rows += int((~valid_coords).sum())
            df = df.loc[valid_coords]

            # Ensure string IDs
            df["PhotoID"] = df["PhotoID"].astype(str)

            # Remove duplicates INSIDE current file
            before_file_dedup = len(df)
            df = df.drop_duplicates(subset="PhotoID", keep="first")
            duplicate_rows += before_file_dedup - len(df)

            # Remove already-seen IDs globally
            mask_new = ~df["PhotoID"].isin(seen_photo_ids)

            duplicate_rows += int((~mask_new).sum())

            df = df.loc[mask_new]

            # Update global ID cache
            seen_photo_ids.update(df["PhotoID"].tolist())

            if len(df) == 0:
                continue

            # Create CSV writer once
            if writer is None:
                writer = csv.writer(out_file)
                writer.writerow(df.columns)

            # MUCH faster than df.to_csv repeatedly
            writer.writerows(df.itertuples(index=False, name=None))

            rows_written += len(df)

            if i % 25 == 0 or i == len(files_to_read):
                print(
                    f"Processed: {i}/{len(files_to_read)} Files | "
                    f"saved Points: {rows_written:,}"
                )

    print("Compiling finished.")
    print(f"Read rows: {rows_read:,}")
    print(f"Saved rows: {rows_written:,}")
    print(f"Duplicates dropped: {duplicate_rows:,}")
    print(f"Invalid coordinates dropped: {invalid_rows:,}")

    return output_csv


compile_raw_flickr_data(flickr_files, COMPILED_CSV)

pd.read_csv(
    COMPILED_CSV,
    nrows=5,
    dtype=MIXED_TYPE_DTYPES,
    low_memory=False
)

In [ ]:
def load_polygon_data(city_path, park_path):
    polygon_records = []

    layers = [
        ("City", city_path, "Geografisc"),
        ("Park", park_path, "NAME"),
    ]

    for poly_type, path, name_field in layers:
        gdf_original = gpd.read_file(path)
        print(f"{poly_type} CRS before: {gdf_original.crs}")

        # GeoPandas is only for importing, transforming, and calculating areas in square meters.
        gdf_lonlat = gdf_original.to_crs(epsg=4326)
        gdf_metric = gdf_original.to_crs(epsg=25832)

        print(f"{poly_type} Bounds aafter EPSG:4326: {tuple(round(v, 5) for v in gdf_lonlat.total_bounds)}")

        for idx, row in gdf_lonlat.iterrows():
            geom = row.geometry
            base_name = row[name_field] if name_field in row and pd.notna(row[name_field]) else f"{poly_type}_{idx}"
            poly_name = f"{poly_type}_{base_name}"
            area_m2 = float(gdf_metric.geometry.iloc[idx].area)

            if geom.geom_type == "Polygon":
                parts = [geom]
            elif geom.geom_type == "MultiPolygon":
                parts = list(geom.geoms)
            else:
                continue

            for part_id, part in enumerate(parts):
                exterior = [(float(x), float(y)) for x, y in part.exterior.coords]
                holes = [[(float(x), float(y)) for x, y in ring.coords] for ring in part.interiors]
                polygon_records.append({
                    "name": poly_name,
                    "type": poly_type,
                    "part_id": part_id,
                    "exterior": exterior,
                    "holes": holes,
                    "area_m2": area_m2,
                    "geometry": part,
                })

    return polygon_records


polygon_records = load_polygon_data(GEOJSON_CITIES, GEOJSON_PARKS)
print(f"Polygon-objects prepared: {len(polygon_records)}")
print(f"Polygone with holes: {sum(1 for p in polygon_records if p['holes'])}")

# 3. Spatial Algorithms

Using classes for the three manual algorithms. The code follows the logics, like: point distances, bounding boxes, line/segment logic, point-in-polygon, and grid-based point indexing.

In [ ]:
class Point():
    # initialise
    def __init__(self, x=None, y=None, pid=None, attrs=None):
        self.x = float(x)
        self.y = float(y)
        self.id = pid
        self.attributes = attrs or {}

    # representation
    def __repr__(self):
        return f"Point(x={self.x}, y={self.y})"

    # test for equality between two points
    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return self.x == other.x and self.y == other.y
    
    '''
    def isEqual(self, other):
        return self.x == other.x and self.y == other.y
    ''' 

    def __hash__(self):
        return hash((self.x, self.y))

    # calculate Euclidean distance between two points
    def distEuclidean(self, other):
        return sqrt((self.x - other.x)**2 + (self.y - other.y)**2)

    # Haversine distance between two lon/lat points, returned in metres
    def distHaversine(self, other):
        r = 6371000
        phi1 = radians(self.y)
        phi2 = radians(other.y)
        lam1 = radians(self.x)
        lam2 = radians(other.x)

        d = 2 * r * arcsin(sqrt(
            sin((phi2 - phi1) / 2)**2 +
            cos(phi1) * cos(phi2) * sin((lam2 - lam1) / 2)**2
        ))
        return float(d)

    # determine position of this point in relation to a vector of two other points
    def sideLine(self, p1, p2):
        side = int((p2.x - p1.x) * (self.y - p1.y) - (self.x - p1.x) * (p2.y - p1.y))
        if side != 0:
            side = side / abs(side)
        return side


class Bbox():
    # initialise
    def __init__(self, data):
        if isinstance(data, Segment):
            x = [data.start.x, data.end.x]
            y = [data.start.y, data.end.y]
        else:
            x = [p.x for p in data]
            y = [p.y for p in data]

        self.ll = Point(min(x), min(y))
        self.ur = Point(max(x), max(y))
        self.ctr = Point((min(x) + max(x)) / 2, (min(y) + max(y)) / 2)
        self.area = abs(max(x) - min(x)) * abs(max(y) - min(y))

    def __repr__(self):
        return f"Bounding box with lower-left {self.ll} and upper-right {self.ur}"

    def testOverlap(self, other):
        if (self.ur.x >= other.ll.x and other.ur.x >= self.ll.x and
            self.ur.y >= other.ll.y and other.ur.y >= self.ll.y):
            return True
        return False

    def containsPoint(self, p):
        if (self.ur.x >= p.x >= self.ll.x and self.ur.y >= p.y >= self.ll.y):
            return True
        return False

    def intersectsRegion(self, other):
        if not self.testOverlap(other):
            return None
        llx = max(self.ll.x, other.ll.x)
        lly = max(self.ll.y, other.ll.y)
        urx = min(self.ur.x, other.ur.x)
        ury = min(self.ur.y, other.ur.y)
        return Bbox([Point(llx, lly), Point(urx, ury)])


class Segment():
    # initialise
    def __init__(self, p0, p1, sid=None):
        self.start = p0
        self.end = p1
        self.sid = sid
        self.length = p0.distEuclidean(p1)

    def __repr__(self):
        return f"Segment with start {self.start} and end {self.end}."

    def intersects(self, other):
        self_bbox = Bbox(self)
        other_bbox = Bbox(other)
        bbox_overlap = self_bbox.testOverlap(other_bbox)

        if bbox_overlap == False:
            return False

        apq = self.start.sideLine(other.start, other.end)
        bpq = self.end.sideLine(other.start, other.end)
        pab = other.start.sideLine(self.start, self.end)
        qab = other.end.sideLine(self.start, self.end)

        if (apq + bpq == 0 and pab + qab == 0):
            return True
        return False


class CoursePolygon():
    # child-style polygon class adapted from the point-in-polygon practical
    def __init__(self, exterior=None, holes=None, name="", poly_type="", area_m2=0):
        self.points = [Point(x, y) for x, y in exterior]
        if self.points[0] != self.points[-1]:
            self.points.append(self.points[0])

        self.holes = []
        for ring in holes or []:
            hole_points = [Point(x, y) for x, y in ring]
            if hole_points and hole_points[0] != hole_points[-1]:
                hole_points.append(hole_points[0])
            self.holes.append(hole_points)

        self.hole_bboxes = [Bbox(h) for h in self.holes]  # pre-computed, free rejection test

        self.size = len(self.points)
        self.name = name
        self.poly_type = poly_type
        self.area_m2 = area_m2
        self.bbox = Bbox(self.points)

    def __repr__(self):
        return f"CoursePolygon(name={self.name}, type={self.poly_type}, points={self.size}, holes={len(self.holes)})"

    def __getitem__(self, key):
        return self.points[key]

    def isClosed(self):
        return self.points[0] == self.points[-1]

    def _ringContainsPoint(self, p, ring_points):
        count = 0
        for i in range(0, len(ring_points) - 1):
            start = ring_points[i]
            end = ring_points[i + 1]

            if (p.y > min(start.y, end.y)):
                if (p.y <= max(start.y, end.y)):
                    if (p.x <= max(start.x, end.x)):
                        if (start.y != end.y):
                            x_intersection = start.x + (p.y - start.y) * (end.x - start.x) / (end.y - start.y)
                            if p.x <= x_intersection:
                                count += 1

        return count % 2 != 0
    '''
    def containsPoint(self, p):
        if self.bbox.containsPoint(p) == False:
            return False

        if self._ringContainsPoint(p, self.points) == False:
            return False

        # Interior rings are holes, so points inside holes are outside the polygon.
        for hole in self.holes:
            if self._ringContainsPoint(p, hole):
                return False
    '''
    # DELETE the existing containsPoint and replace with:
    def containsPoint(self, p):
        # 1. Quick exterior bbox rejection
        if not self.bbox.containsPoint(p):
            return False

        # 2. Ray-casting on exterior ring (Practical 4)
        if not self._ringContainsPoint(p, self.points):
            return False

        # 3. Hole check — bbox pre-filter before expensive ray-casting
        for hole_pts, hole_bbox in zip(self.holes, self.hole_bboxes):
            if hole_bbox.containsPoint(p) and self._ringContainsPoint(p, hole_pts):
                return False  # inside a hole = outside the polygon

        return True

In [ ]:
def build_course_polygons(records):
    polygons = []
    for record in records:
        polygons.append(CoursePolygon(
            exterior=record["exterior"],
            holes=record["holes"],
            name=record["name"],
            poly_type=record["type"],
            area_m2=record["area_m2"]
        ))
    return polygons


all_polygons = build_course_polygons(polygon_records)
print(f"Manuelle Polygone erstellt: {len(all_polygons)}")
print(all_polygons[0])

# A quick PIP test with the polygon holes
sample_polygon = CoursePolygon(
    exterior=[[0, 0], [10, 0], [10, 10], [0, 10], [0, 0]],
    holes=[[[3, 3], [7, 3], [7, 7], [3, 7], [3, 3]]],
    name="Sample",
    poly_type="Test",
    area_m2=100
)
assert sample_polygon.containsPoint(Point(2, 2)) == True
assert sample_polygon.containsPoint(Point(5, 5)) == False
assert sample_polygon.containsPoint(Point(12, 2)) == False
print("PIP-Test passed!")

## 3.1 Spatial Indexing

The Flickr dataset is too large to test every point against every polygon directly. Spatial indexing is used to further speed up the processing: the study area is divided into regular grid cells, and each `Point` is assigned to one cell based on its longitude and latitude.

For this project the grid is used in two steps. First, the compiled Flickr table is filtered with polygon bounding boxes so only points near the selected cities and parks are kept as candidates. Second, those candidates are inserted into a `PointIndex`. When a polygon is processed, the index only reads cells intersecting the polygon bounding box, instead of scanning all Flickr points again. This keeps the extensive point-in-polygon checks focused on relevant candidate points.

The grid index is an acceleration structure only: it does not decide whether a point is inside a polygon. It only reduces the search space before the exact manual point-in-polygon algorithm is applied.

In [ ]:
class PointIndex():
    # initialise the index
    def __init__(self, data, box=None, res=0.05):
        self.res = res
        self.bBox = box if box else Bbox(data)
        w = self.bBox.ur.x - self.bBox.ll.x
        h = self.bBox.ur.y - self.bBox.ll.y
        self.nCols = int(w / self.res) + 1
        self.nRows = int(h / self.res) + 1

        ur = Point(
            self.bBox.ll.x + (self.nCols * self.res),
            self.bBox.ll.y + (self.nRows * self.res)
        )
        self.bBox = Bbox([self.bBox.ll, ur])
        self.maxIndex = (self.nCols * self.nRows) - 1
        self.points = [[0, []] for _ in range(self.maxIndex + 1)]
        self.bigArray = []

        self.addPoints(data)

    def __repr__(self):
        return f"PointIndex(res={self.res}, nCols={self.nCols}, nRows={self.nRows}, points={len(self.bigArray)})"

    def addPoints(self, data):
        for p in data:
            self.addPoint(p)
            self.bigArray.append(p)

    def addPoint(self, p):
        i = self.pointIndex(p)
        if 0 <= i <= self.maxIndex:
            self.points[i][0] += 1
            self.points[i][1].append(p)

    def pointIndex(self, p):
        j = int((p.y - self.bBox.ll.y) / self.res)
        i = int((p.x - self.bBox.ll.x) / self.res)
        return (j * self.nCols) + i

    def regionQuery(self, region):
        query = self.bBox.intersectsRegion(region)
        if query is None:
            return [], 0

        c_start = max(0, int((query.ll.x - self.bBox.ll.x) / self.res))
        c_end = min(self.nCols - 1, int((query.ur.x - self.bBox.ll.x) / self.res))
        r_start = max(0, int((query.ll.y - self.bBox.ll.y) / self.res))
        r_end = min(self.nRows - 1, int((query.ur.y - self.bBox.ll.y) / self.res))

        ps = []
        for r in range(r_start, r_end + 1):
            for c in range(c_start, c_end + 1):
                index = (r * self.nCols) + c
                if 0 <= index <= self.maxIndex and self.points[index][0] > 0:
                    ps.extend(self.points[index][1])

        final = []
        for p in ps:
            if region.containsPoint(p):
                final.append(p)

        return final, len(final)

    def bruteRegionQuery(self, region):
        final = []
        for p in self.bigArray:
            if region.containsPoint(p):
                final.append(p)
        return final, len(final)

    def nearestPoint(self, p, method="haversine"):
        # Idea: start with nearby cells, then expand until candidates exist.
        step = self.res
        max_step = max(self.bBox.ur.x - self.bBox.ll.x, self.bBox.ur.y - self.bBox.ll.y) + self.res
        ps = []

        while len(ps) == 0 and step <= max_step:
            ll = Point(p.x - step, p.y - step)
            ur = Point(p.x + step, p.y + step)
            box = Bbox([ll, ur])
            ps, count = self.regionQuery(box)
            ps = [q for q in ps if q is not p]
            step = step + self.res

        if len(ps) == 0:
            return None, None

        old_count = len(ps)
        nearest_point, nearest_distance = self.__minDist(ps, p, method=method)

        # Re-query a box based on the current nearest distance.
        if method == "haversine":
            lat_buffer = nearest_distance / 111320.0
            lon_buffer = nearest_distance / (111320.0 * max(math.cos(math.radians(p.y)), 0.01))
        else:
            lat_buffer = nearest_distance
            lon_buffer = nearest_distance

        ll = Point(p.x - lon_buffer, p.y - lat_buffer)
        ur = Point(p.x + lon_buffer, p.y + lat_buffer)
        ps2, count = self.regionQuery(Bbox([ll, ur]))
        ps2 = [q for q in ps2 if q is not p]

        if count > old_count and len(ps2) > 0:
            nearest_point, nearest_distance = self.__minDist(ps2, p, method=method)

        return nearest_point, nearest_distance

    def __minDist(self, points, p, method="haversine"):
        best_point = None
        best_distance = float("inf")
        for q in points:
            if method == "haversine":
                d = p.distHaversine(q)
            else:
                d = p.distEuclidean(q)
            if d < best_distance:
                best_point = q
                best_distance = d
        return best_point, best_distance

In [ ]:
def bbox_prefilter_flickr_points(compiled_csv, output_csv, polygons):
    if os.path.exists(output_csv) and not REBUILD_CANDIDATE_CACHE:
        print(f"Load Bounding-Box candidates from {output_csv}...")
        return output_csv

    polygon_bboxes = [(p.bbox.ll.x, p.bbox.ll.y, p.bbox.ur.x, p.bbox.ur.y) for p in polygons]
    total_rows = 0
    total_candidates = 0

    with open(output_csv, "w", encoding="utf-8", newline="") as out_file:
        wrote_header = False
        for chunk_no, chunk in enumerate(pd.read_csv(compiled_csv, chunksize=CSV_CHUNK_SIZE, dtype=MIXED_TYPE_DTYPES, low_memory=False), start=1):
            total_rows += len(chunk)
            lons = chunk["Longitude"].to_numpy(dtype=float)
            lats = chunk["Latitude"].to_numpy(dtype=float)
            mask = np.zeros(len(chunk), dtype=bool)

            for minx, miny, maxx, maxy in polygon_bboxes:
                mask |= ((lons >= minx) & (lons <= maxx) & (lats >= miny) & (lats <= maxy))

            candidates = chunk.loc[mask].copy()
            total_candidates += len(candidates)
            candidates.to_csv(out_file, index=False, header=not wrote_header)
            wrote_header = True

            if chunk_no % 10 == 0:
                print(f"Chunks: {chunk_no} | Rows reviewed: {total_rows:,} | Candidates: {total_candidates:,}")

    print(f"Bounding-Box Pre-Filter completed: {total_rows:,} -> {total_candidates:,} Candidates")
    return output_csv

'''
def point_from_row(row):
    attrs = {col: getattr(row, col) for col in FLICKR_COLUMNS if hasattr(row, col)}
    return Point(row.Longitude, row.Latitude, pid=row.PhotoID, attrs=attrs)
'''

bbox_prefilter_flickr_points(COMPILED_CSV, CANDIDATE_CSV, all_polygons)
candidate_df = pd.read_csv(CANDIDATE_CSV, dtype=MIXED_TYPE_DTYPES, low_memory=False)
'''
spatial_points = [point_from_row(row) for row in candidate_df.itertuples(index=False)]
''' 
# zip() is a C-level iterator — much faster than itertuples().
# Only the DataFrame index is stored in attrs; all other data
# stays in candidate_df and is fetched once at the end of PIP.
def build_points_fast(df):
    spatial_points = []
    for idx, lon, lat, pid in zip(df.index,
                                   df['Longitude'],
                                   df['Latitude'],
                                   df['PhotoID']):
        spatial_points.append(Point(lon, lat, pid=pid, attrs={'df_index': idx}))
    return spatial_points

spatial_points = build_points_fast(candidate_df)

print(f"Candidate points for the Algorithms: {len(spatial_points):,}")
if len(spatial_points) == 0:
    raise ValueError("No candidate points found. Please verify CRS and Polygon-Bounds.")

global_grid = PointIndex(spatial_points, res=GLOBAL_INDEX_RESOLUTION)
print(global_grid)

# Test: Index query vs. brute force on a small subset
subset = spatial_points[:min(5000, len(spatial_points))]
subset_index = PointIndex(subset, res=GLOBAL_INDEX_RESOLUTION)
indexed_points, indexed_count = subset_index.regionQuery(all_polygons[0].bbox)
brute_points, brute_count = subset_index.bruteRegionQuery(all_polygons[0].bbox)
assert indexed_count == brute_count
print(f"Spatial-Index-Test passed: {indexed_count} == {brute_count}")

## 3.2 Point in Polygon

The point-in-polygon algorithm follows the ray-casting logic. For a test point, a horizontal ray is imagined from the point to the right. The algorithm counts how often this ray crosses the polygon boundary. If the number of crossings is odd, the point is inside; if the number is even, it is outside.

Before ray casting, every polygon uses its bounding box as a fast pre-check. Points outside this rectangle cannot be inside the polygon, so they are skipped immediately. The candidate points come from the spatial index, which means the exact ray-casting test is only applied to a much smaller set of points.

The park polygons contain interior rings, or holes. A point must be inside the exterior ring and outside all holes to be counted as inside the park. The notebook therefore checks the exterior first and then excludes points that fall inside any interior ring. The output keeps one row per point-polygon match, which allows city and park results to be summarized independently.

In [ ]:
def _ring_contains_batch(ring_points, xs, ys):
    """
    Vectorized ray-casting for N points simultaneously.
    Same algorithm as CoursePolygon._ringContainsPoint — just numpy instead
    of a Python loop over one point at a time.
    
    ring_points : list of Point  (polygon ring, closed)
    xs, ys      : numpy arrays of candidate coordinates (length N)
    returns     : boolean numpy array, True where point is inside the ring
    """
    count = np.zeros(len(xs), dtype=np.int32)

    for i in range(len(ring_points) - 1):
        x1 = ring_points[i].x;     y1 = ring_points[i].y
        x2 = ring_points[i + 1].x; y2 = ring_points[i + 1].y

        if y1 == y2:          # horizontal edge — skip (same as original)
            continue

        # Conditions mirror _ringContainsPoint exactly, applied to all N points
        cond = (
            (ys >  min(y1, y2)) &
            (ys <= max(y1, y2)) &
            (xs <= max(x1, x2))
        )

        if not np.any(cond):  # fast skip if no candidates pass bbox test
            continue

        # x-coordinate where the ray hits this edge
        x_int = x1 + (ys[cond] - y1) * (x2 - x1) / (y2 - y1)
        count[cond] += (xs[cond] <= x_int).astype(np.int32)

    return (count % 2) != 0


def batch_pip(poly, xs, ys):
    """
    Batch Point-in-Polygon for a CoursePolygon against N candidate points.

    Pipeline (mirrors containsPoint):
      1. Bbox rejection     — numpy comparison, eliminates most candidates instantly
      2. Exterior ring test — vectorized ray-casting (_ring_contains_batch)
      3. Hole tests         — vectorized ray-casting per hole, with bbox pre-filter

    Returns a boolean numpy array of length N.
    """
    # 1. Bbox pre-filter (vectorized)
    inside = (
        (xs >= poly.bbox.ll.x) & (xs <= poly.bbox.ur.x) &
        (ys >= poly.bbox.ll.y) & (ys <= poly.bbox.ur.y)
    )

    if not np.any(inside):
        return inside

    # 2. Exterior ring — only run on bbox survivors
    inside[inside] = _ring_contains_batch(poly.points,
                                          xs[inside], ys[inside])

    if not np.any(inside):
        return inside

    # 3. Holes — same vectorized ray-casting with per-hole bbox pre-filter
    for hole_pts, hole_bbox in zip(poly.holes, poly.hole_bboxes):
        candidates = inside.copy()

        # hole bbox pre-filter (matches containsPoint hole optimization)
        candidates &= (
            (xs >= hole_bbox.ll.x) & (xs <= hole_bbox.ur.x) &
            (ys >= hole_bbox.ll.y) & (ys <= hole_bbox.ur.y)
        )

        if np.any(candidates):
            in_hole = _ring_contains_batch(hole_pts,
                                           xs[candidates], ys[candidates])
            # Points inside a hole are OUTSIDE the polygon
            tmp = candidates.copy()
            tmp[tmp] = in_hole
            inside[tmp] = False

    return inside


def run_point_in_polygon(polygons, point_index, output_csv):
    if os.path.exists(output_csv) and not REBUILD_PIP_RESULTS:
        print(f"Load saved PIP-Results from {output_csv}...")
        return pd.read_csv(output_csv, dtype=MIXED_TYPE_DTYPES, low_memory=False)

    inside_indices = []
    poly_names     = []
    poly_types     = []
    poly_areas     = []

    for i, poly in enumerate(polygons, start=1):
        candidates, candidate_count = point_index.regionQuery(poly.bbox)

        if candidate_count == 0:
            print(f"{i:02d}/{len(polygons)} {poly.name}: Candidates = 0, inside = 0")
            continue

        # Extract coordinates into numpy arrays — one vectorized PIP call
        xs = np.array([p.x for p in candidates], dtype=np.float64)
        ys = np.array([p.y for p in candidates], dtype=np.float64)

        mask = batch_pip(poly, xs, ys)           # replaces the per-point Python loop
        inside_count = int(mask.sum())

        for p in (c for c, m in zip(candidates, mask) if m):
            inside_indices.append(p.attributes['df_index'])
            poly_names.append(poly.name)
            poly_types.append(poly.poly_type)
            poly_areas.append(poly.area_m2)

        print(f"{i:02d}/{len(polygons)} {poly.name}: "
              f"Candidates = {candidate_count:,}, inside = {inside_count:,}")

    result = candidate_df.loc[inside_indices].copy()
    result["polygon_name"]    = poly_names
    result["polygon_type"]    = poly_types
    result["polygon_area_m2"] = poly_areas

    result.to_csv(output_csv, index=False)
    print(f"PIP finished. Saved results: {len(result):,}")
    return result


inside_points_df = run_point_in_polygon(all_polygons, global_grid, FILTERED_CSV)
inside_points_df.head()

In [ ]:
'''
def run_point_in_polygon(polygons, point_index, output_csv):
    if os.path.exists(output_csv) and not REBUILD_PIP_RESULTS:
        print(f"Load saved PIP-Results from {output_csv}...")
        return pd.read_csv(output_csv, dtype=MIXED_TYPE_DTYPES, low_memory=False)

    records = []

    for i, poly in enumerate(polygons, start=1):
        candidates, candidate_count = point_index.regionQuery(poly.bbox)
        inside_count = 0

        for p in candidates:
            if poly.containsPoint(p):
                row = p.attributes.copy()
                row["polygon_name"] = poly.name
                row["polygon_type"] = poly.poly_type
                row["polygon_area_m2"] = poly.area_m2
                records.append(row)
                inside_count += 1

        print(f"{i:02d}/{len(polygons)} {poly.name}: Candidates = {candidate_count:,}, inside = {inside_count:,}")

    output_columns = FLICKR_COLUMNS + ["polygon_name", "polygon_type", "polygon_area_m2"]
    result = pd.DataFrame(records, columns=output_columns)
    result.to_csv(output_csv, index=False)
    print(f"PIP finished. Saved results: {len(result):,}")
    return result
'''

def run_point_in_polygon(polygons, point_index, output_csv):
    if os.path.exists(output_csv) and not REBUILD_PIP_RESULTS:
        print(f"Load saved PIP-Results from {output_csv}...")
        return pd.read_csv(output_csv, dtype=MIXED_TYPE_DTYPES, low_memory=False)

    # Collect only lightweight integers + strings — no dict copies in the loop
    inside_indices = []
    poly_names     = []
    poly_types     = []
    poly_areas     = []

    for i, poly in enumerate(polygons, start=1):
        candidates, candidate_count = point_index.regionQuery(poly.bbox)
        inside_count = 0

        for p in candidates:
            if poly.containsPoint(p):
                inside_indices.append(p.attributes['df_index'])
                poly_names.append(poly.name)
                poly_types.append(poly.poly_type)
                poly_areas.append(poly.area_m2)
                inside_count += 1

        print(f"{i:02d}/{len(polygons)} {poly.name}: "
              f"Candidates = {candidate_count:,}, inside = {inside_count:,}")

    # One vectorized DataFrame lookup at the end — replaces all the dict copies
    result = candidate_df.loc[inside_indices].copy()
    result["polygon_name"]    = poly_names
    result["polygon_type"]    = poly_types
    result["polygon_area_m2"] = poly_areas

    result.to_csv(output_csv, index=False)
    print(f"PIP finished. Saved results: {len(result):,}")
    return result

inside_points_df = run_point_in_polygon(all_polygons, global_grid, FILTERED_CSV)
inside_points_df.head()

In [ ]:
if len(inside_points_df) == 0:
    print("No points found within the polygons.")
else:
    point_counts = (
        inside_points_df
        .groupby(["polygon_type", "polygon_name"])
        .size()
        .reset_index(name="point_count")
        .sort_values(["polygon_type", "point_count"], ascending=[True, False])
    )
    display(point_counts)

## 3.3a Sampled / Thinned Nearest Neighbor Index

This cell keeps the first, faster NNI approach as a comparison method. It creates a random subset of up to `NNI_SAMPLE_SIZE` points per polygon and computes NNI inside that subset. In other words, both the query points and the possible nearest neighbors come from the same thinned sample.

This is useful as a fast approximation and fallback, but it should be interpreted as the NNI of a random 5,000-point thinning of each polygon, not as the exact NNI of the full Flickr point pattern. The next cell adds a more realistic sampled-query NNI that keeps all points available as possible nearest neighbors.

In [ ]:
def df_to_points(df):
    return [point_from_row(row) for row in df.itertuples(index=False)]


def sample_polygon_points(group, max_points=NNI_SAMPLE_SIZE):
    total_points = len(group)
    if total_points > max_points:
        sampled = group.sample(n=max_points, random_state=RANDOM_SEED).copy()
    else:
        sampled = group.copy()
    return sampled, total_points, len(sampled)


def compute_sampled_nni(points_list, area_m2):
    n = len(points_list)
    if n < 2 or area_m2 <= 0:
        return None

    local_index = PointIndex(points_list, res=NNI_INDEX_RESOLUTION)
    distances = []

    for p in points_list:
        nearest_point, nearest_distance = local_index.nearestPoint(p, method="haversine")
        if nearest_distance is not None:
            distances.append(nearest_distance)

    if len(distances) == 0:
        return None

    observed_mean_distance = sum(distances) / len(distances)
    point_density = n / area_m2
    expected_mean_distance = 1.0 / (2.0 * math.sqrt(point_density))
    nni = observed_mean_distance / expected_mean_distance

    return observed_mean_distance, expected_mean_distance, nni


def run_sampled_nni(inside_df, output_csv):
    result_columns = [
        "polygon_type", "polygon_name", "total_points", "nni_points_used", "area_m2",
        "observed_mean_distance_m", "expected_mean_distance_m", "nni", "pattern"
    ]

    if os.path.exists(output_csv) and not REBUILD_NNI_RESULTS:
        print(f"Load saved NNI results from {output_csv}...")
        return pd.read_csv(output_csv, low_memory=False)

    if len(inside_df) == 0:
        empty = pd.DataFrame(columns=result_columns)
        empty.to_csv(output_csv, index=False)
        return empty

    results = []

    for (poly_type, poly_name), group in inside_df.groupby(["polygon_type", "polygon_name"]):
        sampled_group, total_points, nni_points_used = sample_polygon_points(group)
        area_m2 = float(group["polygon_area_m2"].iloc[0])
        sampled_points = df_to_points(sampled_group)

        stats = compute_sampled_nni(sampled_points, area_m2)
        if stats is None:
            observed, expected, nni = np.nan, np.nan, np.nan
            pattern = "not enough points"
        else:
            observed, expected, nni = stats
            pattern = "clustered" if nni < 1 else "dispersed/random"

        results.append({
            "polygon_type": poly_type,
            "polygon_name": poly_name,
            "total_points": total_points,
            "nni_points_used": nni_points_used,
            "area_m2": area_m2,
            "observed_mean_distance_m": observed,
            "expected_mean_distance_m": expected,
            "nni": nni,
            "pattern": pattern,
        })

        print(f"{poly_name}: total = {total_points:,}, used for NNI = {nni_points_used:,}, NNI = {nni:.4f}" if not np.isnan(nni) else f"{poly_name}: not enough points!")

    results_df = pd.DataFrame(results, columns=result_columns).sort_values(["polygon_type", "nni"])
    results_df.to_csv(output_csv, index=False)
    return results_df


nni_results = run_sampled_nni(inside_points_df, NNI_RESULTS_CSV)
nni_results

## 3.3b Sampled-Query Nearest Neighbor Index

The sampled-query NNI is the better version for interpretation. It still samples at most `NNI_SAMPLE_SIZE` query points per polygon to keep runtime manageable, but it builds the local `PointIndex` from all points inside that polygon. Therefore, each sampled query point searches for its nearest neighbor among the full polygon point set.

The expected nearest-neighbor distance is also based on the full point density, using `total_points / polygon_area_m2`. This better reflects the actual Flickr point pattern while avoiding the cost of calculating nearest neighbors for every single point in very dense polygons such as Berlin or Hamburg.

Additional step: spatial deduplication of points that have been stored stacked at the sam ecoordinates, that would falsify the NNI.

In [ ]:
# 3.3b REALISTIC Sampled-Query Nearest Neighbor Index (Spatially Deduplicated & Normalized)

def compute_fast_sampled_query_nni(group, area_m2, sample_size=NNI_SAMPLE_SIZE):
    # 1. SPATIAL DEDUPLICATION: Keep only unique coordinates
    unique_group = group.drop_duplicates(subset=["Latitude", "Longitude"])
    total_points = len(unique_group)
    
    if total_points < 2 or area_m2 <= 0:
        return None

    lats = unique_group["Latitude"].to_numpy(dtype=float)
    lons = unique_group["Longitude"].to_numpy(dtype=float)

    # 2. Select random query indices from the UNIQUE points
    if total_points > sample_size:
        np.random.seed(RANDOM_SEED)
        query_indices = np.random.choice(total_points, sample_size, replace=False)
    else:
        query_indices = np.arange(total_points)

    query_points_used = len(query_indices)

    lats_rad = np.radians(lats)
    lons_rad = np.radians(lons)
    
    distances = np.empty(query_points_used)
    R = 6371000.0 # Earth radius in meters

    for i, q_idx in enumerate(query_indices):
        lat1_rad = lats_rad[q_idx]
        lon1_rad = lons_rad[q_idx]

        dphi = lats_rad - lat1_rad
        dlam = lons_rad - lon1_rad

        a = np.sin(dphi / 2.0)**2 + np.cos(lat1_rad) * np.cos(lats_rad) * np.sin(dlam / 2.0)**2
        c = 2 * np.arcsin(np.sqrt(a))
        dist_array = R * c

        dist_array[q_idx] = np.inf
        distances[i] = np.min(dist_array)

    observed_mean_distance = np.mean(distances)
    point_density = total_points / area_m2
    expected_mean_distance = 1.0 / (2.0 * np.sqrt(point_density))
    nni = observed_mean_distance / expected_mean_distance

    # 3. NORMALIZATION: Calculate Z-score to determine statistical significance
    se = 0.26136 / np.sqrt((total_points**2) / area_m2)
    z_score = (observed_mean_distance - expected_mean_distance) / se

    return observed_mean_distance, expected_mean_distance, nni, z_score, query_points_used, total_points


def run_fast_sampled_query_nni(inside_df, output_csv):
    result_columns = [
        "polygon_type", "polygon_name", "total_photos", "unique_locations", "query_points_used", "area_m2",
        "observed_mean_distance_m", "expected_mean_distance_m", "nni", "z_score", "pattern"
    ]

    if os.path.exists(output_csv) and not REBUILD_NNI_RESULTS:
        print(f"Load saved Sampled Query NNI results from {output_csv}...")
        return pd.read_csv(output_csv, low_memory=False)

    if len(inside_df) == 0:
        empty = pd.DataFrame(columns=result_columns)
        empty.to_csv(output_csv, index=False)
        return empty

    results = []

    for (poly_type, poly_name), group in inside_df.groupby(["polygon_type", "polygon_name"]):
        area_m2 = float(group["polygon_area_m2"].iloc[0])
        
        stats = compute_fast_sampled_query_nni(group, area_m2)

        if stats is None:
            observed, expected, nni, z_score = np.nan, np.nan, np.nan, np.nan
            query_points_used = 0
            unique_locations = 0
            pattern = "not enough points"
        else:
            observed, expected, nni, z_score, query_points_used, unique_locations = stats
            
            # Determine pattern based on both NNI and statistically significant Z-score (< -1.96 or > 1.96)
            if nni < 1 and z_score < -1.96:
                pattern = "clustered"
            elif nni > 1 and z_score > 1.96:
                pattern = "dispersed"
            else:
                pattern = "random"

        results.append({
            "polygon_type": poly_type,
            "polygon_name": poly_name,
            "total_photos": len(group),
            "unique_locations": unique_locations,
            "query_points_used": query_points_used,
            "area_m2": area_m2,
            "observed_mean_distance_m": observed,
            "expected_mean_distance_m": expected,
            "nni": nni,
            "z_score": z_score,
            "pattern": pattern,
        })

        print(
            f"{poly_name}: photos={len(group):,}, unique_locs={unique_locations:,}, "
            f"NNI={nni:.4f}, Z={z_score:.1f}"
            if not np.isnan(nni) else f"{poly_name}: not enough points"
        )

    results_df = pd.DataFrame(results, columns=result_columns).sort_values(["polygon_type", "nni"])
    results_df.to_csv(output_csv, index=False)
    return results_df

# Execute the updated realistic version
nni_query_results = run_fast_sampled_query_nni(inside_points_df, NNI_QUERY_RESULTS_CSV)
display(nni_query_results)

# 4. Summary & Results

The final comparison uses the sampled-query NNI as the prefered result. This version samples query points for speed, but searches nearest neighbors among all points in each polygon and uses the full point density for the expected distance. Values below 1 indicate clustering; values above 1 indicate dispersion/randomness relative to a random point pattern.

The sampled/thinned NNI from the previous cell remains available as a fallback and comparison, but its values describe a random subset rather than the full Flickr point pattern.

Run this cell once, if you need to change the following parts, but don't want to execute the whole notebook again:

In [ ]:

inside_points_df = pd.read_csv(FILTERED_CSV)

nni_query_results = pd.read_csv(NNI_QUERY_RESULTS_CSV)


In [ ]:
recommended_nni = nni_query_results

if len(recommended_nni) == 0:
    print("No Sampled Query NNI results available.")
else:
    summary = (
        recommended_nni
        .dropna(subset=["nni"])
        .groupby("polygon_type")
        .agg(
            polygons=("polygon_name", "count"),
            unique_locations=("unique_locations", "sum"),
            mean_nni=("nni", "mean"),
            median_nni=("nni", "median"),
            mean_observed_distance_m=("observed_mean_distance_m", "mean"),
            mean_expected_distance_m=("expected_mean_distance_m", "mean"),
        )
        .reset_index()
    )
    display(summary)

    if set(summary["polygon_type"]) >= {"City", "Park"}:
        avg_city = float(summary.loc[summary["polygon_type"] == "City", "mean_nni"].iloc[0])
        avg_park = float(summary.loc[summary["polygon_type"] == "Park", "mean_nni"].iloc[0])
        stronger_cluster = "Städten" if avg_city < avg_park else "Parks"
        print(f"Average Sampled-Query NNI by City: {avg_city:.4f}")
        print(f"Average Sampled-Query NNI by Parks:  {avg_park:.4f}")
        print(f"Conclusion: The Flickr points are stronger clustered in {stronger_cluster}!")

    if 'nni_results' in globals() and len(nni_results) > 0:
        comparison = recommended_nni[["polygon_type", "polygon_name", "nni"]].merge(
            nni_results[["polygon_type", "polygon_name", "nni"]],
            on=["polygon_type", "polygon_name"],
            suffixes=("_sampled_query", "_sampled_thinned")
        )
        display(comparison.sort_values(["polygon_type", "nni_sampled_query"]))


In [ ]:
import matplotlib.pyplot as plt

if len(recommended_nni.dropna(subset=["nni"])) > 0:
    plot_df = recommended_nni.dropna(subset=["nni"]).sort_values("nni")
    colors = plot_df["polygon_type"].map({"City": "#1680cd", "Park": "#51aa31"})

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.barh(plot_df["polygon_name"], plot_df["nni"], color=colors)
    ax.axvline(1.0, color="black", linewidth=1, linestyle="--", label="Random pattern (NNI = 1)")
    ax.set_xlabel("Sampled Nearest Neighbor Index")
    ax.set_ylabel("Polygon")
    ax.set_title("Flickr clustering in selected German cities and parks")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt

if len(inside_points_df) > 0:
    cities_plot = gpd.read_file(GEOJSON_CITIES).to_crs(epsg=4326)
    parks_plot = gpd.read_file(GEOJSON_PARKS).to_crs(epsg=4326)

    plot_points = inside_points_df.drop_duplicates("PhotoID")
    if len(plot_points) > 20000:
        plot_points = plot_points.sample(20000, random_state=RANDOM_SEED)

    fig, ax = plt.subplots(figsize=(12, 10))
    parks_plot.boundary.plot(ax=ax, color="#51aa31", linewidth=1, label="Parks")
    cities_plot.boundary.plot(ax=ax, color="#1680cd", linewidth=1, label="Cities")
    ax.scatter(plot_points["Longitude"], plot_points["Latitude"], s=1, alpha=0.25, color="#a10b0b", label="Flickr points")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("Flickr points inside selected polygons")
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

In [ ]:
print("Output-Files:")
print(f"Compiled Flickr-Points: {COMPILED_CSV}")
print(f"Bounding-Box-Candidates: {CANDIDATE_CSV}")
print(f"PIP-Results: {FILTERED_CSV}")
print(f"Sampled/thinned NNI Results: {NNI_RESULTS_CSV}")
print(f"Sampled-query NNI Results: {NNI_QUERY_RESULTS_CSV}")

# 5. Extra: Validation with geospatial libraries

In [ ]:
# --- 5. Validation with Geospatial Libraries ---
#
# Three steps:
# 1. Export point data to GeoPackage files (data/processed/)
#  2. Compute ground-truth NNI using sklearn BallTree (haversine, ALL unique locations)
#   3. Visualise: bar-chart comparison (manual vs library) + NNI choropleth map

from sklearn.neighbors import BallTree
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

PROCESSED_ALL_GPKG      = PATH_PROCESSED / 'all_flickr_points.gpkg'
PROCESSED_FILTERED_GPKG = PATH_PROCESSED / 'filtered_flickr_inside_polygons.gpkg'
VALIDATION_NNI_CSV      = PATH_OUTPUTS / 'nni_results_validation.csv'

# --- 1. Export GeoPackages ---
print("1/3  Export GeoPackage files...")

all_df = pd.read_csv(COMPILED_CSV, dtype=MIXED_TYPE_DTYPES, low_memory=False)
gpd.GeoDataFrame(
    all_df,
    geometry=gpd.points_from_xy(all_df.Longitude, all_df.Latitude),
    crs="EPSG:4326"
).to_file(PROCESSED_ALL_GPKG, driver="GPKG")
print(f"     All Flickr-Points       : {len(all_df):,}  →  {PROCESSED_ALL_GPKG}")

filtered_gdf = gpd.GeoDataFrame(
    inside_points_df,
    geometry=gpd.points_from_xy(inside_points_df.Longitude, inside_points_df.Latitude),
    crs="EPSG:4326"
)
filtered_gdf.to_file(PROCESSED_FILTERED_GPKG, driver="GPKG")
print(f"     Filtered Points (PIP)  : {len(filtered_gdf):,}  →  {PROCESSED_FILTERED_GPKG}")

In [ ]:
# ── 2. Library NNI — BallTree (haversine, no sampling) ────────────────────────
# sklearn BallTree with metric='haversine' operates on (lat, lon) in radians.
# k=2 query: result[:,0] is the point itself (distance=0); result[:,1] is the NN.
# All unique coordinate locations are used — no sampling at all.

def compute_library_nni(group, area_m2):
    unique  = group.drop_duplicates(subset=["Latitude", "Longitude"])
    n       = len(unique)
    if n < 2 or area_m2 <= 0:
        return None

    coords_rad        = np.radians(unique[["Latitude", "Longitude"]].values)
    tree              = BallTree(coords_rad, metric="haversine")
    distances, _      = tree.query(coords_rad, k=2)      # k=2 skips self (dist=0)
    nn_m              = distances[:, 1] * 6371000.0       # radians → metres

    observed  = float(np.mean(nn_m))
    density   = n / area_m2
    expected  = 1.0 / (2.0 * np.sqrt(density))
    nni       = observed / expected
    se        = 0.26136 / np.sqrt(n**2 / area_m2)        # Clark & Evans (1954)
    z_score   = (observed - expected) / se

    return observed, expected, nni, z_score, n


print("\n2/3  Calculate validation NNI (BallTree, all unique coordinates)...")
val_rows = []

for (poly_type, poly_name), group in inside_points_df.groupby(["polygon_type", "polygon_name"]):
    area_m2 = float(group["polygon_area_m2"].iloc[0])
    stats   = compute_library_nni(group, area_m2)

    if stats is None:
        obs, exp, nni, z, n_uniq = np.nan, np.nan, np.nan, np.nan, 0
        pattern = "not enough points"
    else:
        obs, exp, nni, z, n_uniq = stats
        if   nni < 1 and z < -1.96: pattern = "clustered"
        elif nni > 1 and z >  1.96: pattern = "dispersed"
        else:                        pattern = "random"

    val_rows.append({
        "polygon_type":             poly_type,
        "polygon_name":             poly_name,
        "total_photos":             len(group),
        "unique_locations":         n_uniq,
        "area_m2":                  area_m2,
        "observed_mean_distance_m": obs,
        "expected_mean_distance_m": exp,
        "nni":                      nni,
        "z_score":                  z,
        "pattern":                  pattern,
    })
    print(f"     {poly_name:<45}  unique={n_uniq:>7,}  NNI={nni:.4f}  Z={z:>7.1f}  [{pattern}]")

val_df = (pd.DataFrame(val_rows)
            .sort_values(["polygon_type", "nni"])
            .reset_index(drop=True))
val_df.to_csv(VALIDATION_NNI_CSV, index=False)
print(f"\n     Results saved: {VALIDATION_NNI_CSV}")
display(val_df[["polygon_type", "polygon_name", "unique_locations", "nni", "z_score", "pattern"]])

# ── 3. Plots ───────────────────────────────────────────────────────────────────
print("\n3/3  Create validation plots...")

TYPE_COLORS = {"City": "#1b7bc0", "Park": "#52a02f"}

# Merge manual (cell 18, sampled) with library (all points) for comparison
compare = (
    nni_query_results[["polygon_name", "polygon_type", "nni"]]
    .rename(columns={"nni": "nni_manual"})
    .dropna(subset=["nni_manual"])
    .merge(
        val_df[["polygon_name", "nni"]].rename(columns={"nni": "nni_library"}),
        on="polygon_name"
    )
    .sort_values("nni_library")
    .reset_index(drop=True)
)

# ── Plot A + B: comparison bar chart  &  scatter agreement ────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17, 7))

# — Bar chart (grouped) —
x  = np.arange(len(compare))
w  = 0.38
c_dark  = [TYPE_COLORS[t] for t in compare["polygon_type"]]
c_light = ["#aec7e8" if t == "City" else "#98df8a" for t in compare["polygon_type"]]

ax1.bar(x - w/2, compare["nni_manual"],  width=w, color=c_dark,  label="Manual (sampled, 5 000 queries)")
ax1.bar(x + w/2, compare["nni_library"], width=w, color=c_light,
        edgecolor="dimgray", linewidth=0.5, label="Library BallTree (all unique locations)")
ax1.axhline(1.0, color="black", linestyle="--", linewidth=1.2, label="NNI = 1 (random)")
ax1.set_xticks(x)
ax1.set_xticklabels(compare["polygon_name"], rotation=45, ha="right", fontsize=8)
ax1.set_ylabel("NNI")
ax1.set_title("Manual vs. Library NNI\n(lower = more clustered)", fontsize=11, fontweight="bold")
ax1.set_ylim(0, max(compare[["nni_manual", "nni_library"]].max()) * 1.2)
ax1.grid(axis="y", alpha=0.3)
handles = [
    mpatches.Patch(color="#1f77b4", label="City — manual"),
    mpatches.Patch(color="#aec7e8", edgecolor="dimgray", label="City — library"),
    mpatches.Patch(color="#2ca02c", label="Park — manual"),
    mpatches.Patch(color="#98df8a", edgecolor="dimgray", label="Park — library"),
    plt.Line2D([0], [0], color="black", linestyle="--", label="NNI = 1"),
]
ax1.legend(handles=handles, fontsize=8, framealpha=0.9)

# — Scatter (agreement plot) —
for ptype, marker in [("City", "o"), ("Park", "^")]:
    sub = compare[compare["polygon_type"] == ptype]
    ax2.scatter(sub["nni_manual"], sub["nni_library"],
                color=TYPE_COLORS[ptype], marker=marker, s=80, zorder=3, label=ptype)
    for _, row in sub.iterrows():
        label = row["polygon_name"].replace("City_", "").replace("Park_", "")[:13]
        ax2.annotate(label, (row["nni_manual"], row["nni_library"]),
                     textcoords="offset points", xytext=(5, 3), fontsize=7, alpha=0.85)

all_nni = pd.concat([compare["nni_manual"], compare["nni_library"]])
lo, hi  = all_nni.min() * 0.85, all_nni.max() * 1.1
ax2.plot([lo, hi], [lo, hi], "k--", linewidth=1.2, label="1:1 reference")
ax2.set_xlim(lo, hi); ax2.set_ylim(lo, hi)
ax2.set_xlabel("Manual NNI  (sampled queries)")
ax2.set_ylabel("Library NNI  (all unique locations)")
ax2.set_title("Agreement: Manual vs. Library\n(points on diagonal = perfect agreement)",
              fontsize=11, fontweight="bold")
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PATH_OUTPUTS / "validation_nni_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Map: NNI choropleth ────────────────────────────────────────────────────────
# Load polygon GeoJSONs and build polygon_name column to match val_df
'''
cities_gdf = gpd.read_file(GEOJSON_CITIES).to_crs(epsg=4326)
parks_gdf  = gpd.read_file(GEOJSON_PARKS).to_crs(epsg=4326)

cities_gdf["polygon_name"] = "City_" + cities_gdf["Geografisc"].astype(str)
parks_gdf["polygon_name"]  = "Park_" + parks_gdf["NAME"].astype(str)

cities_plot = cities_gdf.merge(val_df[["polygon_name", "nni", "pattern"]], on="polygon_name", how="left")
parks_plot  = parks_gdf.merge( val_df[["polygon_name", "nni", "pattern"]], on="polygon_name", how="left")

nni_min = val_df["nni"].min()
nni_max = val_df["nni"].max()
cmap    = plt.cm.RdYlGn_r                             # green=clustered, red=dispersed
norm    = Normalize(vmin=nni_min * 0.9, vmax=nni_max * 1.05)

fig, ax = plt.subplots(figsize=(10, 11))

# Flickr points as faint background
bg_pts = inside_points_df.drop_duplicates("PhotoID")
if len(bg_pts) > 15000:
    bg_pts = bg_pts.sample(15000, random_state=RANDOM_SEED)
ax.scatter(bg_pts.Longitude, bg_pts.Latitude,
           s=1, alpha=0.12, color="steelblue", zorder=1, label="Flickr points (sample)")

# Polygon fills colour-coded by NNI
cities_plot.plot(column="nni", ax=ax, cmap=cmap, norm=norm,
                 edgecolor="white", linewidth=1.0, zorder=2)
parks_plot.plot(column="nni", ax=ax, cmap=cmap, norm=norm,
                edgecolor="white", linewidth=1.0, zorder=2,
                hatch="///", alpha=0.9)

# NNI labels inside each polygon
for _, row in cities_plot.dropna(subset=["nni"]).iterrows():
    cx, cy = row.geometry.centroid.x, row.geometry.centroid.y
    name   = str(row["Geografisc"])[:9]
    ax.annotate(f"{name}\n{row['nni']:.3f}", (cx, cy),
                ha="center", fontsize=6.5, fontweight="bold",
                color="white", zorder=5)
for _, row in parks_plot.dropna(subset=["nni"]).iterrows():
    cx, cy = row.geometry.centroid.x, row.geometry.centroid.y
    name   = str(row["NAME"])[:12]
    ax.annotate(f"{name}\n{row['nni']:.3f}", (cx, cy),
                ha="center", fontsize=6, color="white", zorder=5)

# Colorbar
sm   = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.028, pad=0.02)
cbar.set_label("NNI (BallTree, alle eindeutigen Standorte)", fontsize=9)
cbar.ax.text(1.45, 0.02, "Geclustert", transform=cbar.ax.transAxes, va="bottom", fontsize=8)
cbar.ax.text(1.45, 0.98, "Dispersed",  transform=cbar.ax.transAxes, va="top",    fontsize=8)

# Legend for polygon types
city_p = mpatches.Patch(facecolor="gray", edgecolor="white", label="Stadt")
park_p = mpatches.Patch(facecolor="gray", edgecolor="white", hatch="///", label="Naturpark (schraffiert)")
pt_p   = plt.Line2D([0],[0], marker="o", color="w", markerfacecolor="steelblue",
                    markersize=6, alpha=0.5, label="Flickr-Punkte (15 k)")
ax.legend(handles=[city_p, park_p, pt_p], loc="upper right", fontsize=9, framealpha=0.9)

ax.set_title(
    "Validation: NNI per polygon (BallTree, all unique coordinates)\n"
    "Green = clustered  |  Red = dispersed",
    fontsize=12, fontweight="bold"
)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
plt.tight_layout()
plt.savefig(PATH_OUTPUTS / "validation_nni_map.png", dpi=150, bbox_inches="tight")
plt.show()
'''

print("\nValidation completed.")
'''
print(f"  Comparison chart : ./outputs/validation_nni_comparison.png")
print(f"  Map          : ./outputs/validation_nni_map.png")
print(f"  CSV            : {VALIDATION_NNI_CSV}")
print(f"  GeoPackage (all Points)      : {PROCESSED_ALL_GPKG}")
print(f"  GeoPackage (filtered Points): {PROCESSED_FILTERED_GPKG}")
'''

print(f"  Comparison plot : {PATH_OUTPUTS / 'validation_nni_comparison.png'}")
print(f"  Map             : {PATH_OUTPUTS / 'validation_nni_map.png'}")
print(f"  CSV             : {VALIDATION_NNI_CSV}")
print(f"  GeoPackage (all points)      : {PROCESSED_ALL_GPKG}")
print(f"  GeoPackage (filtered points) : {PROCESSED_FILTERED_GPKG}")

In [ ]:
import folium
from folium.plugins import MarkerCluster
import branca.colormap as cm
import geopandas as gpd
import pandas as pd

# 1. Prepare the Polygon Data
# Load GeoJSONs and standardize names to match nni_query_results
cities_gdf = gpd.read_file(GEOJSON_CITIES).to_crs(epsg=4326)
parks_gdf = gpd.read_file(GEOJSON_PARKS).to_crs(epsg=4326)

cities_gdf["polygon_name"] = "City_" + cities_gdf["Geografisc"].astype(str)
parks_gdf["polygon_name"] = "Park_" + parks_gdf["NAME"].astype(str)

# Combine polygons and merge with NNI results
combined_gdf = pd.concat([cities_gdf, parks_gdf], ignore_index=True)
map_gdf = combined_gdf.merge(
    nni_query_results[['polygon_name', 'nni', 'pattern']], 
    on='polygon_name', 
    how='left'
)

# --- ADD THIS FIX ---
# Convert any Timestamp/datetime columns into strings so Folium can serialize them
for col in map_gdf.columns:
    if col != 'geometry':
        # Check if the column is a datetime type
        if pd.api.types.is_datetime64_any_dtype(map_gdf[col]):
            map_gdf[col] = map_gdf[col].astype(str)
        # Also catch hidden timestamps inside generic 'object' columns
        elif map_gdf[col].dtype == 'object':
            map_gdf[col] = map_gdf[col].apply(lambda x: str(x) if isinstance(x, pd.Timestamp) else x)
# --------------------

# 2. Initialize the Folium Map (Centered on Germany)
m = folium.Map(location=[51.1657, 10.4515], zoom_start=6, tiles=None)

# 3. Add Basemaps
folium.TileLayer('cartodbpositron', name='CartoDB Positron').add_to(m)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Esri Satellite',
    overlay=False
).add_to(m)

# 4. Add Polygon Layer colored by NNI
# Create a Linear Colormap (Red for clustered/low NNI, Green for dispersed/high NNI)
min_nni = map_gdf['nni'].min()
max_nni = map_gdf['nni'].max()
colormap = cm.LinearColormap(colors=['red', 'orange', 'yellow', 'lightgreen'], vmin=min_nni, vmax=max_nni)
colormap.caption = 'Sampled-Query Nearest Neighbor Index (NNI)'

def style_fn(feature):
    nni_val = feature['properties'].get('nni')
    color = colormap(nni_val) if pd.notna(nni_val) else 'gray'
    return {
        'fillColor': color,
        'color': 'black',
        'weight': 1,
        'fillOpacity': 0.65
    }

# Create a FeatureGroup for polygons
poly_layer = folium.FeatureGroup(name='Polygons (by NNI)')
folium.GeoJson(
    map_gdf,
    style_function=style_fn,
    tooltip=folium.GeoJsonTooltip(
        fields=['polygon_name', 'nni', 'pattern'], 
        aliases=['Location:', 'NNI Score:', 'Pattern:']
    )
).add_to(poly_layer)

poly_layer.add_to(m)
m.add_child(colormap)

# 5. Add Flickr Points Layer (with Clustering for performance)
point_layer = folium.FeatureGroup(name='Flickr Points (Sampled)', show=False)
marker_cluster = MarkerCluster().add_to(point_layer)

# Sample points to avoid crashing the browser DOM
plot_points = inside_points_df.drop_duplicates(subset=["PhotoID"])
if len(plot_points) > 10000:
    plot_points = plot_points.sample(n=10000, random_state=RANDOM_SEED)

for idx, row in plot_points.iterrows():
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=3,
        color='#1680cd',
        fill=True,
        fill_opacity=0.7,
        tooltip=f"Photo ID: {row.get('PhotoID', 'N/A')}"
    ).add_to(marker_cluster)

point_layer.add_to(m)

# 6. Add Layer Control and Display
folium.LayerControl(position='topright').add_to(m)

# Display map
m